# Partie 3 — Spark SQL et DataFrames
Prérequis : `trips_history.jsonl` dans `lake/bronze/history/` et `stations.json` dans `lake/reference/`.

In [1]:
from common import get_spark
from schemas import trip_schema, station_schema
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = get_spark("Pedali-Part3-DataFrames")
sc = spark.sparkContext
print(spark.version, "| cœurs :", sc.defaultParallelism)

3.5.3 | cœurs : 2


## 1. Contrôle du format de stations.json
Les noms de champs attendus : station_id, name, city, district, zone_type, capacity, latitude, longitude.
Si les vôtres diffèrent, modifiez `schemas.py` (puis *Kernel → Restart*).

In [2]:
!head -c 700 /home/jovyan/data/stations.json

{"station_id": "S-CAS-001", "name": "Gare Casa-Voyageurs", "city": "Casablanca", "district": "Belvédère", "zone_type": "gare", "capacity": 40, "latitude": 33.5893, "longitude": -7.5906}
{"station_id": "S-CAS-002", "name": "Gare Casa-Port", "city": "Casablanca", "district": "Centre-ville", "zone_type": "gare", "capacity": 36, "latitude": 33.5997, "longitude": -7.6134}
{"station_id": "S-CAS-003", "name": "Gare L'Oasis", "city": "Casablanca", "district": "Oasis", "zone_type": "gare", "capacity": 28, "latitude": 33.556, "longitude": -7.628}
{"station_id": "S-CAS-004", "name": "Mosquée Hassan II", "city": "Casablanca", "district": "Ancienne Médina", "zone_type": "tourisme", "capacity": 30, "l

## 2. Lecture avec schéma explicite (sans inférence)

In [3]:
TRIPS_PATH    = "s3a://lake/bronze/history/trips_history.jsonl"
STATIONS_PATH = "s3a://lake/reference/stations.json"

trips_raw = spark.read.schema(trip_schema).json(TRIPS_PATH)

stations = spark.read.schema(station_schema).json(STATIONS_PATH)
if stations.filter("station_id IS NOT NULL").count() == 0:
    # le fichier est un tableau JSON sur plusieurs lignes
    stations = spark.read.schema(station_schema).option("multiLine", True).json(STATIONS_PATH)

trips = (trips_raw
    .dropDuplicates(["trip_id", "status"])
    .withColumn("event_time", F.to_timestamp("event_time")))

trips.printSchema()
print("Trajets (dédoublonnés) :", trips.count())
print("Stations               :", stations.count(), "(attendu : 400)")
stations.show(5, truncate=False)

root
 |-- trip_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- start_station: string (nullable = true)
 |-- end_station: string (nullable = true)
 |-- city: string (nullable = true)
 |-- status: string (nullable = true)
 |-- duration_s: integer (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- battery_pct: integer (nullable = true)
 |-- price_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

Trajets (dédoublonnés) : 49040
Stations               : 400 (attendu : 400)
+----------+-------------------+----------+---------------+---------------+--------+--------+---------+
|station_id|name               |city      |district       |zone_type      |capacity|latitude|longitude|
+----------+-------------------+----------+---------------+---------------+--------+--------+---------+
|S-CAS-001 |Gare Casa-Voyageurs|Casablanca|Belvédère      |gare 

In [4]:
timings = {}

def timed(label, df, n=20):
    t0 = time.time()
    df.show(n, truncate=False)
    timings[label] = round(time.time() - t0, 2)
    print(f"⏱  {label} : {timings[label]}s")

def bench(label, df):
    t0 = time.time()
    df.write.format("noop").mode("overwrite").save()
    timings[label] = round(time.time() - t0, 2)
    print(f"⏱  {label} : {timings[label]}s")

## 3. Chiffre d'affaires par ville : API DataFrame puis SQL
Les résultats doivent être **identiques** à ceux de la Partie 2.

In [5]:
ended = trips.filter(F.col("status") == "ENDED").cache()
print("ENDED :", ended.count())

ca_df = (ended.groupBy("city")
    .agg(F.round(F.sum("price_mad"), 2).alias("ca_total_mad"),
         F.count("*").alias("nb_trajets"))
    .orderBy(F.desc("ca_total_mad")))
timed("DF - CA par ville", ca_df)

ENDED : 15612
+----------+------------+----------+
|city      |ca_total_mad|nb_trajets|
+----------+------------+----------+
|Casablanca|111883.46   |5843      |
|Marrakech |58128.99    |3568      |
|Rabat     |56013.93    |3378      |
|Tanger    |45385.42    |2823      |
+----------+------------+----------+

⏱  DF - CA par ville : 31.16s


In [6]:
ended.createOrReplaceTempView("ended_trips")

ca_sql = spark.sql("""
    SELECT city,
           ROUND(SUM(price_mad), 2) AS ca_total_mad,
           COUNT(*)                 AS nb_trajets
    FROM ended_trips
    GROUP BY city
    ORDER BY ca_total_mad DESC
""")
timed("SQL - CA par ville", ca_sql)

+----------+------------+----------+
|city      |ca_total_mad|nb_trajets|
+----------+------------+----------+
|Casablanca|111883.46   |5843      |
|Marrakech |58128.99    |3568      |
|Rabat     |56013.93    |3378      |
|Tanger    |45385.42    |2823      |
+----------+------------+----------+

⏱  SQL - CA par ville : 8.49s


## 4. Durée moyenne et top 5 en DataFrame (comparaison avec la Partie 2)

In [7]:
duree_df = (ended.groupBy("city", "vehicle_type")
    .agg(F.round(F.avg("duration_s") / 60, 1).alias("duree_moy_min"),
         F.count("*").alias("nb_trajets"))
    .orderBy("city", "vehicle_type"))
timed("DF - durée moyenne", duree_df, n=30)

top5_df = (ended.groupBy("start_station").count()
    .orderBy(F.desc("count")).limit(5))
timed("DF - top 5 stations", top5_df)

+----------+------------+-------------+----------+
|city      |vehicle_type|duree_moy_min|nb_trajets|
+----------+------------+-------------+----------+
|Casablanca|BIKE        |27.9         |1758      |
|Casablanca|EBIKE       |21.2         |2351      |
|Casablanca|SCOOTER     |22.9         |1734      |
|Marrakech |BIKE        |25.0         |1100      |
|Marrakech |EBIKE       |17.5         |1479      |
|Marrakech |SCOOTER     |17.9         |989       |
|Rabat     |BIKE        |23.5         |999       |
|Rabat     |EBIKE       |17.5         |1346      |
|Rabat     |SCOOTER     |18.9         |1033      |
|Tanger    |BIKE        |23.7         |850       |
|Tanger    |EBIKE       |16.8         |1131      |
|Tanger    |SCOOTER     |17.9         |842       |
+----------+------------+-------------+----------+

⏱  DF - durée moyenne : 6.71s
+-------------+-----+
|start_station|count|
+-------------+-----+
|S-MAR-002    |166  |
|S-MAR-004    |156  |
|S-MAR-005    |150  |
|S-CAS-001    |144  |

## 5. Jointure avec le référentiel stations (zone, quartier)

In [8]:
st = stations.select(
    F.col("station_id").alias("start_station"),
    F.col("name").alias("start_station_name"),
    F.col("district").alias("start_district"),
    F.col("zone_type"),
    F.col("capacity").alias("start_capacity"))

joined = ended.join(st, on="start_station", how="left")
print("Trajets sans station connue :", joined.filter(F.col("zone_type").isNull()).count())

zone_kpi = (joined.groupBy("city", "zone_type")
    .agg(F.count("*").alias("nb_trajets"),
         F.round(F.sum("price_mad"), 2).alias("ca_mad"))
    .orderBy("city", F.desc("nb_trajets")))
timed("DF - KPI par zone et ville", zone_kpi, n=60)

Trajets sans station connue : 0
+----------+---------------+----------+--------+
|city      |zone_type      |nb_trajets|ca_mad  |
+----------+---------------+----------+--------+
|Casablanca|centre_affaires|2276      |42361.69|
|Casablanca|residentiel    |1665      |38668.18|
|Casablanca|commercial     |839       |11724.97|
|Casablanca|tourisme       |443       |6720.35 |
|Casablanca|gare           |345       |6069.66 |
|Casablanca|plage          |178       |3850.33 |
|Casablanca|universite     |97        |2488.28 |
|Marrakech |tourisme       |1200      |18358.51|
|Marrakech |residentiel    |886       |18401.32|
|Marrakech |centre_affaires|864       |12983.58|
|Marrakech |universite     |242       |3838.66 |
|Marrakech |gare           |200       |2467.1  |
|Marrakech |commercial     |176       |2079.82 |
|Rabat     |centre_affaires|1076      |18406.52|
|Rabat     |residentiel    |1041      |19417.57|
|Rabat     |commercial     |347       |4544.95 |
|Rabat     |tourisme       |330      

## 6. UDF trip_band contre fonctions natives (when / otherwise)
court < 10 min ; moyen de 10 à 30 min ; long > 30 min.

In [9]:
def trip_band_py(d):
    if d is None:
        return None
    if d < 600:
        return "court"
    if d <= 1800:
        return "moyen"
    return "long"

trip_band_udf = F.udf(trip_band_py, StringType())
spark.udf.register("trip_band", trip_band_py, StringType())

d = F.col("duration_s")
trip_band_native = (F.when(d < 600, "court")
                     .when(d <= 1800, "moyen")
                     .when(d > 1800, "long"))

with_udf    = ended.withColumn("trip_band", trip_band_udf(F.col("duration_s")))
with_native = ended.withColumn("trip_band", trip_band_native)

bench("warmup", with_native)
bench("UDF Python (1)", with_udf);    bench("UDF Python (2)", with_udf)
bench("Natif when (1)", with_native); bench("Natif when (2)", with_native)

with_native.groupBy("city", "trip_band").count().orderBy("city", "trip_band").show(20)

⏱  warmup : 10.62s
⏱  UDF Python (1) : 37.94s
⏱  UDF Python (2) : 30.21s
⏱  Natif when (1) : 6.55s
⏱  Natif when (2) : 11.84s
+----------+---------+-----+
|      city|trip_band|count|
+----------+---------+-----+
|Casablanca|    court| 1581|
|Casablanca|     long| 1698|
|Casablanca|    moyen| 2564|
| Marrakech|    court| 1053|
| Marrakech|     long|  748|
| Marrakech|    moyen| 1767|
|     Rabat|    court| 1042|
|     Rabat|     long|  714|
|     Rabat|    moyen| 1622|
|    Tanger|    court| 1038|
|    Tanger|     long|  569|
|    Tanger|    moyen| 1216|
+----------+---------+-----+



In [10]:
print("=== Plan UDF (cherchez BatchEvalPython) ===")
with_udf.explain()
print("=== Plan natif (CASE WHEN dans la JVM) ===")
with_native.explain()

=== Plan UDF (cherchez BatchEvalPython) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#58, pythonUDF0#4765 AS trip_band#2874]
   +- BatchEvalPython [trip_band_py(duration_s#8)#2873], [pythonUDF0#4765]
      +- InMemoryTableScan [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#58]
            +- InMemoryRelation [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#58], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- AdaptiveSparkPlan isFinalPlan=true
                     +- == Final Plan ==
                        SortAggregate(

## 7. Couche Silver : Parquet partitionné par city

In [11]:
silver = (trips
    .join(st, on="start_station", how="left")
    .withColumn("trip_band", trip_band_native))

(silver.write.mode("overwrite").partitionBy("city")
    .parquet("s3a://lake/silver/trips_history/"))

check = spark.read.parquet("s3a://lake/silver/trips_history/")
check.groupBy("city").count().show()
print("Lignes Silver :", check.count())

+----------+-----+
|      city|count|
+----------+-----+
|Casablanca|18391|
|     Rabat|10590|
|    Tanger| 8892|
| Marrakech|11167|
+----------+-----+

Lignes Silver : 49040


## 8. explain(True) sur la jointure
Repérer : Parsed / Analyzed / Optimized Logical Plan, puis Physical Plan avec **BroadcastHashJoin**.

In [12]:
plan_df = (trips.filter(F.col("status") == "ENDED")
    .join(st, on="start_station", how="left")
    .groupBy("city", "zone_type")
    .agg(F.count("*").alias("nb_trajets"), F.sum("price_mad").alias("ca_mad")))

plan_df.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city, 'zone_type], ['city, 'zone_type, count(1) AS nb_trajets#5315L, sum('price_mad) AS ca_mad#5317]
+- Project [start_station#4, trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#58, start_station_name#2123, start_district#2124, zone_type#30, start_capacity#2125]
   +- Join LeftOuter, (start_station#4 = start_station#2122)
      :- Filter (status#7 = ENDED)
      :  +- Project [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, to_timestamp(event_time#12, None, TimestampType, Some(UTC), false) AS event_time#58]
      :     +- Deduplicate [trip_id#0, status#7]
      :        +- Relation [trip_id#0,user_id#1,vehicle_id#2,vehicle_type#3,start_station#4,end_station#5,city#6,status#7,duration_s#8,distance_km#9,battery_pct#10,price_mad#11,ev

In [13]:
# Forcer un SortMergeJoin pour comparer
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
plan_df.explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#6, zone_type#30], functions=[count(1), sum(price_mad#11)])
   +- Exchange hashpartitioning(city#6, zone_type#30, 200), ENSURE_REQUIREMENTS, [plan_id=1309]
      +- HashAggregate(keys=[city#6, zone_type#30], functions=[partial_count(1), partial_sum(price_mad#11)])
         +- Project [city#6, price_mad#11, zone_type#30]
            +- BroadcastHashJoin [start_station#4], [start_station#2122], LeftOuter, BuildRight, false
               :- InMemoryTableScan [start_station#4, city#6, price_mad#11]
               :     +- InMemoryRelation [trip_id#0, user_id#1, vehicle_id#2, vehicle_type#3, start_station#4, end_station#5, city#6, status#7, duration_s#8, distance_km#9, battery_pct#10, price_mad#11, event_time#58], StorageLevel(disk, memory, deserialized, 1 replicas)
               :           +- AdaptiveSparkPlan isFinalPlan=true
                              +- == Final Plan ==
                             

## 9. Récapitulatif des temps

In [14]:
print(f"{'Mesure':<32}{'Temps (s)':>10}")
for k, v in timings.items():
    print(f"{k:<32}{v:>10}")

Mesure                           Temps (s)
DF - CA par ville                    31.16
SQL - CA par ville                    8.49
DF - durée moyenne                    6.71
DF - top 5 stations                  18.61
DF - KPI par zone et ville           15.03
warmup                               10.62
UDF Python (1)                       37.94
UDF Python (2)                       30.21
Natif when (1)                        6.55
Natif when (2)                       11.84


📸 Captures : explain(True) (avec BroadcastHashJoin visible) ; Spark UI → onglet SQL/DataFrame + Stages ; MinIO → lake/silver/trips_history/city=... .
Puis **Kernel → Shut Down Kernel**.